In [1]:
!pip install flask flask-cors pydub websocket-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [flask-cors]


In [1]:
# pip install pyaudioop

ERROR: Could not find a version that satisfies the requirement pyaudioop (from versions: none)
ERROR: No matching distribution found for pyaudioop
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==========================================
# 文件名: app.py

import os
import ssl
import re
import json
import base64
import hmac
import hashlib
import datetime
import time
import subprocess  # 新增：用于直接调用 ffmpeg
import _thread as thread
from urllib.parse import urlencode
from wsgiref.handlers import format_date_time
from datetime import datetime
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
# from pydub import AudioSegment  <-- 删除了这一行，不再需要它
import websocket

# ========== 1. 请填写你的讯飞 Key ==========
APP_ID = "86282236"
API_KEY = "9179d9ccf5d06d18cce3e6ef034b8571"
API_SECRET = "NGQ2YWU3Y2MxNzJjMDgyYTllMzU3MjAz"

# ========== 2. Flask 设置 ==========
app = Flask(__name__, static_folder='.', static_url_path='')
CORS(app)

# ========== 3. 讯飞 ASR (语音转写) 逻辑 ==========
class WsParamASR:
    def __init__(self, APPID, APIKey, APISecret):
        self.APPID = APPID
        self.APIKey = APIKey
        self.APISecret = APISecret
        self.CommonArgs = {"app_id": self.APPID}
        self.BusinessArgs = {"domain": "iat", "language": "zh_cn", "accent": "mandarin", "vinfo":1,"vad_eos":10000}

    def create_url(self):
        url = 'wss://iat-api.xfyun.cn/v2/iat'
        now = datetime.now()
        date = format_date_time(time.mktime(now.timetuple()))
        signature_origin = "host: " + "iat-api.xfyun.cn" + "\n"
        signature_origin += "date: " + date + "\n"
        signature_origin += "GET " + "/v2/iat " + "HTTP/1.1"
        signature_sha = hmac.new(self.APISecret.encode('utf-8'), signature_origin.encode('utf-8'), digestmod=hashlib.sha256).digest()
        signature_sha = base64.b64encode(signature_sha).decode(encoding='utf-8')
        authorization_origin = f'api_key="{self.APIKey}", algorithm="hmac-sha256", headers="host date request-line", signature="{signature_sha}"'
        authorization = base64.b64encode(authorization_origin.encode('utf-8')).decode(encoding='utf-8')
        v = {"authorization": authorization, "date": date, "host": "iat-api.xfyun.cn"}
        return url + '?' + urlencode(v)

global_asr_result = ""

def run_asr_client(audio_path):
    global global_asr_result
    global_asr_result = ""
    wsParam = WsParamASR(APP_ID, API_KEY, API_SECRET)
    
    def on_message(ws, message):
        global global_asr_result
        try:
            msg = json.loads(message)
            code = msg["code"]
            if code != 0:
                print(f"ASR Error: {code}")
            else:
                data = msg["data"]["result"]["ws"]
                result = ""
                for i in data:
                    for w in i["cw"]:
                        result += w["w"]
                global_asr_result += result
        except Exception as e:
            print("ASR Message Error:", e)

    def on_error(ws, error): print("ASR WS Error:", error)
    def on_close(ws, a, b): pass

    def on_open(ws):
        def run(*args):
            frameSize = 8000
            intervel = 0.04
            status = 0
            # 注意：如果 ffmpeg 转换失败，这里打开文件可能会报错，需要确保 ffmpeg 安装正确
            try:
                with open(audio_path, "rb") as fp:
                    while True:
                        buf = fp.read(frameSize)
                        if not buf: status = 2
                        
                        if status == 0:
                            d = {"common": wsParam.CommonArgs,
                                 "business": wsParam.BusinessArgs,
                                 "data": {"status": 0, "format": "audio/L16;rate=16000",
                                          "audio": str(base64.b64encode(buf), 'utf-8'),
                                          "encoding": "raw"}}
                            ws.send(json.dumps(d))
                            status = 1
                        elif status == 1:
                            d = {"data": {"status": 1, "format": "audio/L16;rate=16000",
                                          "audio": str(base64.b64encode(buf), 'utf-8'),
                                          "encoding": "raw"}}
                            ws.send(json.dumps(d))
                        elif status == 2:
                            d = {"data": {"status": 2, "format": "audio/L16;rate=16000",
                                          "audio": str(base64.b64encode(buf), 'utf-8'),
                                          "encoding": "raw"}}
                            ws.send(json.dumps(d))
                            time.sleep(1)
                            break
                        time.sleep(intervel)
                ws.close()
            except Exception as e:
                print(f"读取音频文件失败: {e}")
                ws.close()

        thread.start_new_thread(run, ())

    websocket.enableTrace(False)
    ws = websocket.WebSocketApp(wsParam.create_url(), on_message=on_message, on_error=on_error, on_close=on_close)
    ws.on_open = on_open
    ws.run_forever(sslopt={"cert_reqs": ssl.CERT_NONE})
    return global_asr_result

# ========== 4. 讯飞 TTS (语音合成) 逻辑 ==========
class WsParamTTS:
    def __init__(self, APPID, APIKey, APISecret, Text):
        self.APPID = APPID
        self.APIKey = APIKey
        self.APISecret = APISecret
        self.Text = Text
        self.CommonArgs = {"app_id": self.APPID}
        self.BusinessArgs = {"aue": "lame", "sfl": 1, "auf": "audio/L16;rate=16000", "vcn": "x2_SuhCn_XiXi", "tte": "utf8"}
        self.Data = {"status": 2, "text": str(base64.b64encode(self.Text.encode('utf-8')), "UTF8")}

    def create_url(self):
        url = 'wss://tts-api.xfyun.cn/v2/tts'
        now = datetime.now()
        date = format_date_time(time.mktime(now.timetuple()))
        signature_origin = "host: " + "ws-api.xfyun.cn" + "\n"
        signature_origin += "date: " + date + "\n"
        signature_origin += "GET " + "/v2/tts " + "HTTP/1.1"
        signature_sha = hmac.new(self.APISecret.encode('utf-8'), signature_origin.encode('utf-8'), digestmod=hashlib.sha256).digest()
        signature_sha = base64.b64encode(signature_sha).decode(encoding='utf-8')
        authorization_origin = f'api_key="{self.APIKey}", algorithm="hmac-sha256", headers="host date request-line", signature="{signature_sha}"'
        authorization = base64.b64encode(authorization_origin.encode('utf-8')).decode(encoding='utf-8')
        v = {"authorization": authorization, "date": date, "host": "ws-api.xfyun.cn"}
        return url + '?' + urlencode(v)

def run_tts_client(text, output_file):
    wsParam = WsParamTTS(APP_ID, API_KEY, API_SECRET, text)
    def on_message(ws, message):
        try:
            msg = json.loads(message)
            if msg["code"] != 0: print("TTS Error:", msg["message"])
            else:
                data = msg["data"]["audio"]
                audio = base64.b64decode(data)
                with open(output_file, 'ab') as f: f.write(audio)
                if msg["data"]["status"] == 2: ws.close()
        except: pass
    ws = websocket.WebSocketApp(wsParam.create_url(), on_message=on_message, on_open=lambda ws: ws.send(json.dumps({"common": wsParam.CommonArgs, "business": wsParam.BusinessArgs, "data": wsParam.Data})))
    ws.run_forever(sslopt={"cert_reqs": ssl.CERT_NONE})

# ========== 5. Web 接口路由 ==========

@app.route('/')
def index():
    return send_file('index.html')

# @app.route('/api/recognize', methods=['POST'])
# def api_recognize():
#     """接收浏览器录音 -> 直接调用 ffmpeg 转 PCM -> 讯飞识别"""
#     if 'audio' not in request.files: return jsonify({'error': 'No audio'}), 400
#     file = request.files['audio']
    
#     ts = str(int(time.time()))
#     webm_path = f"temp_{ts}.webm"
#     pcm_path = f"temp_{ts}.pcm"
    
#     try:
#         file.save(webm_path)
        
#         # [修改] 使用 subprocess 直接调用 ffmpeg，彻底绕过 pydub/audioop
#         # 讯飞要求: 采样率16000, 单声道(ac 1), s16le格式
#         cmd = [
#             'ffmpeg', '-y',          # -y 覆盖输出文件
#             '-i', webm_path,         # 输入文件
#             '-ac', '1',              # 单声道
#             '-ar', '16000',          # 16k 采样率
#             '-f', 's16le',           # PCM 格式 (signed 16-bit little endian)
#             pcm_path                 # 输出文件
#         ]
        
#         # 执行命令，如果报错会捕获异常
#         subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        
#         # 识别
    #     text = run_asr_client(pcm_path)
        
    #     # 清理
    #     try:
    #         os.remove(webm_path)
    #         os.remove(pcm_path)
    #     except: pass
        
    #     return jsonify({'text': text})
        
    # except subprocess.CalledProcessError as e:
    #     print(f"FFmpeg Error: {e}")
    #     return jsonify({'error': 'Audio conversion failed. Is FFmpeg installed?'}), 500
    # except Exception as e:
    #     print(f"General Error: {e}")
    #     return jsonify({'error': str(e)}), 500


@app.route('/api/recognize', methods=['POST'])
def api_recognize():
    """接收浏览器录音 -> 直接调用 ffmpeg 转 PCM -> 讯飞识别"""
    if 'audio' not in request.files: return jsonify({'error': 'No audio'}), 400
    file = request.files['audio']
    
    ts = str(int(time.time()))
    webm_path = f"temp_{ts}.webm"
    pcm_path = f"temp_{ts}.pcm"
    
    try:
        file.save(webm_path)
        
        # 使用 subprocess 直接调用 ffmpeg
        cmd = [
            'ffmpeg', '-y',
            '-i', webm_path,
            '-ac', '1',
            '-ar', '16000',
            '-f', 's16le',
            pcm_path
        ]
        
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        
        # 1. 获取原始识别结果
        text = run_asr_client(pcm_path)
        
        # ========== [新增] 核心过滤逻辑 ==========
        # 使用正则表达式，将所有 a-z 和 A-Z 的英文字母替换为空
        # 这样即使方言被误识别成英文，也会被删掉
        if text:
            text = re.sub(r'[a-zA-Z]', '', text)
            # 去除可能多余的空格
            text = text.strip()
        # =======================================
        
        # 清理临时文件
        try:
            os.remove(webm_path)
            os.remove(pcm_path)
        except: pass
        
        return jsonify({'text': text})
        
    except subprocess.CalledProcessError as e:
        print(f"FFmpeg Error: {e}")
        return jsonify({'error': 'Audio conversion failed. Is FFmpeg installed?'}), 500
    except Exception as e:
        print(f"General Error: {e}")
        return jsonify({'error': str(e)}), 500



@app.route('/api/synthesize', methods=['POST'])
def api_synthesize():
    data = request.json
    text = data.get('text', '')
    if not text: return jsonify({'error': 'No text'}), 400
    
    ts = str(int(time.time()))
    mp3_path = f"tts_{ts}.mp3"
    
    try:
        run_tts_client(text, mp3_path)
        return send_file(mp3_path, mimetype="audio/mpeg")
    except Exception as e:
        print(f"TTS Error: {e}")
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    if not os.path.exists('images'): os.makedirs('images')
    print("----------------------------------------------------------------")
    print("🚀 服务已修复！无需 pydub 即可运行。请访问: http://127.0.0.1:5000")
    print("⚠️ 请确保你的电脑已经安装了 FFmpeg 并在命令行可以运行 'ffmpeg -version'")
    print("----------------------------------------------------------------")
    app.run(port=5000)

----------------------------------------------------------------
🚀 服务已修复！无需 pydub 即可运行。请访问: http://127.0.0.1:5000
⚠️ 请确保你的电脑已经安装了 FFmpeg 并在命令行可以运行 'ffmpeg -version'
----------------------------------------------------------------
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET /images/生成游戏小程序主界面地图.png HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET /images/生成古风戏台图片.png HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET /images/medals/medal-actor.png HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET /images/medals/medal-ambassador.png HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET /images/medals/medal-master.png HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 18:24:28] "GET /videos/sapling.mp4 HTTP/1.1" 206 -
